# Classificação do perfil de investimento

MLP própria em PyTorch para `investmentPolicy`. Métrica principal: F1 macro. Critério: pelo menos 15% acima do melhor baseline no teste.

## 0. Dados, pipeline e separação

Os JSONs são lidos diretamente. O dataset analítico é construído pelo módulo `build_dataset.py`; o pré-processamento é ajustado somente no treino.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EDA_DIR = Path('../eda')
DATA_DIR = EDA_DIR / 'Dados'
sys.path.insert(0, str(Path('code').resolve()))
sys.path.insert(0, str((EDA_DIR / 'code').resolve()))

from build_dataset import build_dataset, BUCKETS
from preprocessing import build_preprocessor
from classification_pipeline import *

fix_seed(SEED)
print('Dispositivo:', DEVICE)

Dispositivo: cpu


In [2]:
df = build_dataset(DATA_DIR)
classes = ['Conservadora', 'Moderada', 'Sofisticada']
label_to_id = dict(zip(classes, range(len(classes))))
target = 'investmentPolicy'

numeric_features = list(BUCKETS.values())
numeric_features += ['num_active_buckets', 'max_bucket_pct']
numeric_features = [column for column in numeric_features if column in df.columns and df[column].nunique() > 1]
categorical_features = ['personType']
feature_columns = numeric_features + categorical_features
df = df[feature_columns + [target]].dropna(subset=[target])
X = df[feature_columns]
y = df[target].map(label_to_id).astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.20, stratify=y_train_full, random_state=SEED)

preprocessor = build_preprocessor(numeric_features, categorical_features)
X_train_array = preprocessor.fit_transform(X_train).astype('float32')
X_val_array = preprocessor.transform(X_val).astype('float32')
X_test_array = preprocessor.transform(X_test).astype('float32')

print('Treino:', X_train_array.shape)
print('Validação:', X_val_array.shape)
print('Teste:', X_test_array.shape)
display(df[target].value_counts(normalize=True).rename('proporção'))

Treino: (12162, 16)
Validação: (3041, 16)
Teste: (3801, 16)


investmentPolicy
Moderada        0.527836
Conservadora    0.431909
Sofisticada     0.040255
Name: proporção, dtype: float64

## 1. Baselines

Classe majoritária e regressão logística balanceada, avaliadas na validação.

In [3]:
dummy, linear, baseline_validation = fit_baselines(X_train_array, y_train, X_val_array, y_val, SEED)
display(baseline_validation)

,modelo,accuracy,f1_macro,roc_auc_ovr_macro
1,Regressão logística,0.401184,0.305494,0.594009
0,Classe majoritária,0.527787,0.230306,NaN


## 2. MLP e treinamento

A MLP tem três logits, `CrossEntropyLoss`, pesos de classe, AdamW, parada antecipada e restauração dos melhores pesos. A implementação completa está em `code/classification_pipeline.py`.

In [4]:
counts = np.bincount(y_train, minlength=len(classes))
class_weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32, device=DEVICE)
base_config = {'hidden_sizes': (64, 32), 'dropout': 0.0, 'learning_rate': 1e-3, 'weight_decay': 0.0, 'batch_size': 128, 'patience': 15, 'max_epochs': 150}
base_model = MLP(X_train_array.shape[1], hidden_sizes=base_config['hidden_sizes'])
print('Parâmetros:', count_parameters(base_model))
base_model, base_history, base_epoch = train_model(base_config, X_train_array, y_train, X_val_array, y_val, class_weights, DEVICE)
display(base_history.tail())

Parâmetros: 3267


,epoch,train_loss,val_loss,val_f1_macro
11,12,1.037285,1.078346,0.344592
12,13,1.036004,1.076100,0.286449
13,14,1.034890,1.081590,0.312082
14,15,1.034890,1.083819,0.318987
15,16,1.032814,1.078925,0.311805


## 3. Regularização e hiperparâmetros

O código abaixo deve ser executado com as oito configurações definidas em `classification_pipeline.py`/nesta seção. A seleção usa apenas F1 macro de validação e compara dropout e weight decay.

In [5]:
configurations = [
    {'name': 'base', 'hidden_sizes': (64, 32), 'dropout': 0.0, 'learning_rate': 1e-3, 'weight_decay': 0.0, 'batch_size': 128},
    {'name': 'dropout_10', 'hidden_sizes': (64, 32), 'dropout': 0.1, 'learning_rate': 1e-3, 'weight_decay': 0.0, 'batch_size': 128},
    {'name': 'dropout_30', 'hidden_sizes': (64, 32), 'dropout': 0.3, 'learning_rate': 1e-3, 'weight_decay': 0.0, 'batch_size': 128},
    {'name': 'weight_decay_1e4', 'hidden_sizes': (64, 32), 'dropout': 0.0, 'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 128},
    {'name': 'weight_decay_1e3', 'hidden_sizes': (64, 32), 'dropout': 0.0, 'learning_rate': 1e-3, 'weight_decay': 1e-3, 'batch_size': 128},
    {'name': 'wide', 'hidden_sizes': (128, 64), 'dropout': 0.2, 'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 128},
    {'name': 'deep', 'hidden_sizes': (128, 64, 32), 'dropout': 0.2, 'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 128},
    {'name': 'small_lr', 'hidden_sizes': (64, 32), 'dropout': 0.2, 'learning_rate': 3e-4, 'weight_decay': 1e-4, 'batch_size': 64},
]
rows = []
models = {}
histories = {}
for config in configurations:
    model, history, best_epoch = train_model(config, X_train_array, y_train, X_val_array, y_val, class_weights, DEVICE)
    predictions, _ = predict_model(model, X_val_array, DEVICE)
    models[config['name']] = model
    histories[config['name']] = history
    rows.append({'configuração': config['name'], 'f1_macro_validação': f1_score(y_val, predictions, average='macro'), 'acurácia_validação': accuracy_score(y_val, predictions), 'melhor_época': best_epoch, 'parâmetros': count_parameters(model)})
search_table = pd.DataFrame(rows).sort_values('f1_macro_validação', ascending=False).reset_index(drop=True)
display(search_table)

,configuração,f1_macro_validação,acurácia_validação,melhor_época,parâmetros
0,wide,0.324759,0.429793,4,10627
1,small_lr,0.311329,0.416310,4,3267
2,deep,0.302021,0.420256,3,12611
3,base,0.297878,0.371259,1,3267
4,weight_decay_1e4,0.297878,0.371259,1,3267
5,weight_decay_1e3,0.297878,0.371259,1,3267
6,dropout_30,0.278701,0.400197,18,3267
7,dropout_10,0.277657,0.351200,1,3267


## 4. Teste final, três sementes e análise de erros

A melhor configuração é retreinada e avaliada uma vez no teste. Depois são calculadas matriz de confusão, métricas por classe, três sementes, erro por subgrupo e importância por permutação.

In [6]:
best_name = search_table.iloc[0]['configuração']
best_config = next(config for config in configurations if config['name'] == best_name)
final_model, final_history, final_epoch = train_model(best_config, X_train_array, y_train, X_val_array, y_val, class_weights, DEVICE)
test_predictions, test_probabilities = predict_model(final_model, X_test_array, DEVICE)

test_table = pd.DataFrame([
    {'modelo': 'Classe majoritária', **metric_values(y_test, dummy.predict(X_test_array))},
    {'modelo': 'Regressão logística', **metric_values(y_test, linear.predict(X_test_array), linear.predict_proba(X_test_array))},
    {'modelo': 'MLP escolhida', **metric_values(y_test, test_predictions, test_probabilities)},
])
display(test_table)
print(classification_report(y_test, test_predictions, target_names=classes, digits=3))

matrix = confusion_matrix(y_test, test_predictions)
print(matrix)

seed_rows = []
for seed in [42, 43, 44]:
    model, _, _ = train_model(best_config, X_train_array, y_train, X_val_array, y_val, class_weights, DEVICE, seed)
    predictions, _ = predict_model(model, X_test_array, DEVICE)
    seed_rows.append({'semente': seed, 'f1_macro': f1_score(y_test, predictions, average='macro'), 'acurácia': accuracy_score(y_test, predictions)})
seed_table = pd.DataFrame(seed_rows)
display(seed_table)

errors = X_test.copy()
errors['real'] = y_test.map(dict(enumerate(classes))).to_numpy()
errors['predito'] = pd.Series(test_predictions, index=X_test.index).map(dict(enumerate(classes))).to_numpy()
errors['erro'] = errors['real'] != errors['predito']
display(errors.groupby('personType', dropna=False)['erro'].agg(['mean', 'count']))
display(errors.groupby('real')['erro'].agg(['mean', 'count']))

,modelo,accuracy,f1_macro,roc_auc_ovr_macro
0,Classe majoritária,0.527756,0.230297,NaN
1,Regressão logística,0.400947,0.308304,0.598199
2,MLP escolhida,0.420942,0.322871,0.595729


              precision    recall  f1-score   support

Conservadora      0.498     0.748     0.598      1642
    Moderada      0.558     0.156     0.243      2006
 Sofisticada      0.076     0.386     0.127       153

    accuracy                          0.421      3801
   macro avg      0.377     0.430     0.323      3801
weighted avg      0.513     0.421     0.392      3801

[[1229  228  185]
 [1165  312  529]
 [  75   19   59]]


,semente,f1_macro,acurácia
0,42,0.322871,0.420942
1,43,0.291578,0.379637
2,44,0.332177,0.425941


,mean,count
personType,,
F,0.570000,3600
J,0.746193,197
NaN,0.500000,4


,mean,count
real,,
Conservadora,0.251523,1642
Moderada,0.844467,2006
Sofisticada,0.614379,153


## 5. Conclusão

Preencher com os valores das tabelas: melhor baseline, arquitetura e parâmetros, época de parada, MLP no teste média ± desvio, critério de sucesso e subgrupo com maior erro. Discutir limitações, desbalanceamento, impacto dos erros e importância por permutação.